### Check number of images for each mushroom

In [ ]:
import os

parent_directory = "./data/archive/data/data"

below = []

for item in os.listdir(parent_directory):
    item_path = os.path.join(parent_directory, item)
    
    if os.path.isdir(item_path):
        file_count = len([
            f for f in os.listdir(item_path) 
            if os.path.isfile(os.path.join(item_path, f))
        ])
        print(f"{item} | {file_count}")
        if file_count < 10:
            below.append((item, file_count))

for folder, count in below:
    print(f"UH OH {folder} only has: {count} files")

### Check bit depth / possible colors

In [ ]:
from PIL import Image

img = Image.open("data/archive/data/data/almond_mushroom/0.png")

mode_info = {
    '1': (1, 2),                  # 1-bit, 2 colors (black and white)
    'L': (8, 256),                # 8-bit, 256 shades of grayscale
    'P': (8, 256),                # 8-bit, palette-based (up to 256 colors)
    'RGB': (24, 16777216),        # 24-bit true color (8 bits x 3 channels)
    'RGBA': (32, 4294967296),     # 32-bit true color with transparency (8 bits x 4 channels)
    'I': (32, "32-bit integer"),
    'F': (32, "32-bit float")
}

bits, possible_colors = mode_info.get(img.mode, (None, "Unknown"))

print(f"Image Mode: {img.mode}")
print(f"Bit Depth: {bits} bits")
print(f"Total Possible Colors: {possible_colors}")

### Full loop with different resolutions and bit depths / possible colors

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import time

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

resolutions = [(64, 64), (128, 128), (256, 256), (512, 512)]
color_modes = ['RGBA', 'RGB', 'P', 'L', '1']

class SimulateColorMode(object):
    """
    Custom transform to simulate different bit depths or color modes.
    """
    def __init__(self, target_mode):
        self.target_mode = target_mode

    def __call__(self, img):
        reduced_img = img.convert(self.target_mode)
        return reduced_img.convert('RGB')

for res in resolutions:
    for color_mode in color_modes:
        print(f"Starting {res} and {color_mode}")
        transform = transforms.Compose([
            transforms.Resize(res),
            SimulateColorMode(target_mode=color_mode),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        dataset = datasets.ImageFolder(root='data/archive/data/data', transform=transform)

        # Remove classes with fewer than 10 images
        class_counts = {}
        for label in dataset.targets:
            class_counts[label] = class_counts.get(label, 0) + 1
        valid_classes = {k for k, v in class_counts.items() if v >= 10}
        dataset.samples = [s for s in dataset.samples if s[1] in valid_classes]
        dataset.targets = [s[1] for s in dataset.samples]
        dataset.imgs = dataset.samples

        train_indices, test_indices = train_test_split(
            range(len(dataset)),
            test_size=0.2,
            stratify=dataset.targets,
            random_state=42
        )

        train_dataset = Subset(dataset, train_indices)
        test_dataset = Subset(dataset, test_indices)

        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

        # Freeze convolutional layers
        for param in model.features.parameters():
            param.requires_grad = False

        num_classes = len(dataset.classes) 
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

        model = model.to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        num_epochs = 15

        # Training
        mean_epoch_time = 0
        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0

            start_time = time.time()
            
            for images, labels in train_loader:
                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                
                loss = criterion(outputs, labels)
                loss.backward()
                
                optimizer.step()
                running_loss += loss.item()
            epoch_loss = running_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
            
            end_time = time.time()

            print(f"Epoch time: {end_time - start_time:.2f} seconds")
            mean_epoch_time += end_time - start_time

        mean_epoch_time /= num_epochs
        print(f"Mean epoch time: {mean_epoch_time:.2f} seconds")

        # Save
        torch.save(model.state_dict(), f"models/{res}_{color_mode}.pth")

        # Evaluation
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                
                outputs = model(images)
                
                _, predicted = torch.max(outputs.data, 1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Test Accuracy: {accuracy:.2f}%")

### Model computational costs

In [ ]:
import numpy as np
from thop import profile

def benchmark(model, input_shape, device="cuda", warmup_iters=50, test_iters=200):
    model.eval()
    model.to(device)
    dummy_input = torch.randn(input_shape).to(device)

    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
    flops = macs * 2

    if device == "cuda":
        with torch.no_grad():
            for _ in range(warmup_iters):
                _ = model(dummy_input)
        torch.cuda.synchronize()

        torch.cuda.reset_peak_memory_stats(device)

        start_events = [torch.cuda.Event(enable_timing=True) for _ in range(test_iters)]
        end_events = [torch.cuda.Event(enable_timing=True) for _ in range(test_iters)]

        with torch.no_grad():
            for i in range(test_iters):
                start_events[i].record()
                _ = model(dummy_input)
                end_events[i].record()

        torch.cuda.synchronize()

        timings = np.array([s.elapsed_time(e) for s, e in zip(start_events, end_events)])
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

        mean_latency = np.mean(timings)
        std_latency = np.std(timings)
    else:
        mean_latency, std_latency, peak_vram_mb = 0.0, 0.0, 0.0

    return {
        "Params (M)": params / 1e6,
        "MACs (G)": macs / 1e9,
        "FLOPs (G)": flops / 1e9,
        "Latency Mean (ms)": mean_latency,
        "Latency Std (ms)": std_latency,
        "Peak VRAM (MB)": peak_vram_mb
    }

# Run for all models
for res in resolutions:
    model = models.efficientnet_v2_s()
    num_classes = len(dataset.classes)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    state_dict = torch.load(f"models/{res}_{color_mode}.pth", weights_only=True)
    model.load_state_dict(state_dict)

    model.to("cuda")
    model.eval()

    results = benchmark(model, input_shape=(1, 3, res[0], res[1]), device="cuda")
    print(results)